In [17]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import os

In [18]:
load_dotenv(find_dotenv())

True

In [19]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

In [20]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="Sentiment of the review.")

structured_sentiment_model = model.with_structured_output(SentimentSchema)

In [21]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description="The type of issue reported in the review.")
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description="The tone of the review.")
    urgency: Literal["high", "medium", "low"] = Field(description="The urgency of the issue reported in the review.")

structured_diagnosis_model = model.with_structured_output(DiagnosisSchema)

In [22]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnoses: dict
    response: str

In [23]:
def find_sentiment(state: ReviewState):
    prompt = f'For the following review, find out the sentiment. \n {state["review"]}'
    sentiment = structured_sentiment_model.invoke(prompt).sentiment
    return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state["sentiment"] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"

def positive_response(state: ReviewState):
    prompt = f"""Write a warm thank you message, in response to this review: 
    \n \n {state['review']} \n Also ask the user to leave a feedback on our website."""

    response = model.invoke(prompt).content
    return {'response': response}

def run_diagnosis(state: ReviewState):
    prompt = f"""For the following review, find out the issue type, tone, and urgency of the issue. 
    \n \n {state['review']}"""
    
    diagnoses = structured_diagnosis_model.invoke(prompt)
    return {'diagnoses': diagnoses.model_dump()}

def negative_response(state: ReviewState):
    prompt = f"""Write a warm thank you message, in response to this review: 
    \n \n {state['review']} \n Also ask the user to leave a feedback on our website. 
    \n Also, write an empathetic, helpful response. 
    \n The issue type is: {state['diagnoses']['issue_type']}. 
    \n The tone is: {state['diagnoses']['tone']}. 
    \n The urgency is: {state['diagnoses']['urgency']}."""

    response = model.invoke(prompt).content
    return {'response': response}

In [27]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment", find_sentiment)
graph.add_node("positive_response", positive_response)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)


graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges("find_sentiment", check_sentiment)
graph.add_edge("positive_response", END)
graph.add_edge("run_diagnosis", "negative_response")
graph.add_edge("negative_response", END)

workflow = graph.compile()

In [29]:
positive_initial_state = {
    "review": "I love this product! It has made my life so much easier and I can't imagine going back to how things were before. The customer service was also fantastic, very responsive and helpful. Highly recommend to anyone looking for a reliable solution.",
}

workflow.invoke(positive_initial_state)

{'review': "I love this product! It has made my life so much easier and I can't imagine going back to how things were before. The customer service was also fantastic, very responsive and helpful. Highly recommend to anyone looking for a reliable solution.",
 'sentiment': 'positive',
 'response': "Dear [Customer's Name],\n\nThank you so much for your wonderful review! We’re thrilled to hear that our product has made such a positive impact on your life. It’s our goal to provide solutions that truly make a difference, and your feedback reaffirms that we’re on the right track.\n\nWe also appreciate your kind words about our customer service team! They work hard to assist our customers, and we'll be sure to share your feedback with them.\n\nIf you have a moment, we’d love for you to leave your thoughts on our website as well. Your insights can help others find the reliable solution they need!\n\nThank you once again for your support and trust in us. We look forward to serving you in the fut

In [30]:
negative_initial_state = {
    "review": "The product was really good, but the customer service was terrible. I had to wait for hours to get a response, and when I finally did, they were unhelpful and rude. This has left me very frustrated and disappointed. I expected better support for a product of this quality.",
}

workflow.invoke(negative_initial_state)

{'review': 'The product was really good, but the customer service was terrible. I had to wait for hours to get a response, and when I finally did, they were unhelpful and rude. This has left me very frustrated and disappointed. I expected better support for a product of this quality.',
 'sentiment': 'negative',
 'diagnoses': {'issue_type': 'Support',
  'tone': 'frustrated',
  'urgency': 'high'},
 'response': "Subject: Thank You for Your Feedback\n\nDear [Customer's Name],\n\nThank you for taking the time to share your experience with us. We truly appreciate your kind words about the product, but we're extremely sorry to hear about the frustration you faced with our customer service. It's disheartening to know we didn't meet your expectations, and I want to sincerely apologize for the long wait and the unhelpful interaction you encountered.\n\nYour feedback is invaluable, and we are committed to improving our support team to ensure that all our customers receive the assistance they dese